# 02 — Train baseline + hierarchical models (multiclass outcomes)

This notebook trains:
- A **baseline multiclass boosted model** (LightGBM) on 4 classes.
- A **hierarchical model** (Stage1 NoDementia vs Dementia; Stage2 subtype among dementia cases) and combines probabilities.

It also saves feature importance and subgroup metrics.

In [2]:

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

# Optional (only used if installed)
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception as e:
    HAS_LGBM = False
    print("LightGBM not available; install lightgbm to run boosted models. Error:", repr(e))

RANDOM_STATE = 42

OUT_DIR = "../output"
FIG_DIR = os.path.join(OUT_DIR, "figs_hierarchical")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

COHORT_PATH = os.path.join(OUT_DIR, "cohort.csv")  # produced by 01_build_cohort
df = pd.read_csv(COHORT_PATH, low_memory=False)

print("Loaded cohort rows:", len(df))
print(df["label"].value_counts(dropna=False))


Loaded cohort rows: 38341
label
Censored         23300
NoDementia_Hy    11083
AD                3068
MixedOrOther       421
PD_LBD             297
Vascular           172
Name: count, dtype: int64


In [3]:

# Keep only the 4 non-censored classes for modeling
KEEP = ["NoDementia_Hy", "AD", "PD_LBD", "Vascular"]
df4 = df[df["label"].isin(KEEP)].copy()
df4["label"] = df4["label"].astype(str)

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

# Separate labels and features
#drop_cols = [c for c in ["NACCID", "baseline_date", "label", "NACCUDSD"] if c in df4.columns]
drop_cols = [c for c in [ "NACCID", "baseline_date", "label",  "has_H_followup", "has_dementia_H", "max_follow_years", "abs_days_from_baseline"] if c in df4.columns]


y = df4["label"].astype(str)
X_all = df4.drop(columns=drop_cols)


4-class rows: 14620
label
NoDementia_Hy    11083
AD                3068
PD_LBD             297
Vascular           172
Name: count, dtype: int64


In [4]:

# --- Treat certain coded vars as categorical (even if stored as ints) ---
CAT_CODE_VARS = [c for c in ["SEX","RACE","HISPANIC"] if c in X_all.columns]

# Standard missing codes
STD_MISS = {-4, 9}

# Clean missing codes and preserve NaNs
for c in CAT_CODE_VARS:
    s = pd.to_numeric(X_all[c], errors="coerce")
    s = s.mask(s.isin(list(STD_MISS)), np.nan)
    # RACE sometimes uses 99 for unknown
    if c == "RACE":
        s = s.mask(s.isin([99]), np.nan)
    X_all[c] = s

# Convert to pandas nullable Int + string so OneHot treats as categorical but NaNs stay NaN
for c in CAT_CODE_VARS:
    X_all[c] = X_all[c].astype("Int64").astype("string")

# Education: keep numeric, plus an education bin for subgroup analysis (optional)
if "EDUC" in X_all.columns:
    educ_num = pd.to_numeric(X_all["EDUC"], errors="coerce")
    X_all["EDUC_BIN"] = pd.Series(np.where(educ_num <= 12, "<=12", ">12"), index=X_all.index).astype("string")

# Check what scikit sees as categorical vs numeric
cat_cols = [c for c in X_all.columns if (X_all[c].dtype == "object") or str(X_all[c].dtype).startswith("string")]
num_cols = [c for c in X_all.columns if c not in cat_cols]

print("Cat cols:", cat_cols)
print("Num cols:", len(num_cols))


Cat cols: ['SEX', 'RACE', 'HISPANIC', 'EDUC_BIN']
Num cols: 53


In [5]:

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
)

print("Train rows:", len(X_train), "Test rows:", len(X_test))


Train rows: 11696 Test rows: 2924


In [5]:
# print("CWD:", os.getcwd())
# print("HAS_LGBM:", HAS_LGBM, "HAS_SHAP:", HAS_SHAP)
# print("FIG_DIR (relative):", FIG_DIR)
# print("FIG_DIR (absolute):", os.path.abspath(FIG_DIR))


In [6]:

import numpy as np
import pandas as pd

# -----------------------------
# Baseline multiclass boosted model (Option B)
# -----------------------------
custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0,
}


def make_sample_weight(y_series, weight_map, default_weight=1.0):
    # map -> float array, and fill any unknown labels safely
    w = y_series.map(weight_map)

    # If any labels missing in the map, fill them (and print them once)
    missing = y_series[w.isna()].unique()
    if len(missing) > 0:
        print("WARNING: labels missing from weight_map; using default_weight for:", list(missing))
        w = w.fillna(default_weight)

    # convert to clean numpy float array (no pd.NA)
    return np.asarray(w, dtype=np.float64)

w_train = make_sample_weight(y_train, custom_w, default_weight=1.0)

# sanity checks (optional but recommended)
assert len(w_train) == len(y_train)
assert np.isfinite(w_train).all()
# w_train = make_sample_weight(y_train, custom_w)

# Label-encode for LightGBM to avoid "Invalid classes" error
label_order = ["AD","NoDementia_Hy","PD_LBD","Vascular"]
label_to_int = {lab:i for i,lab in enumerate(label_order)}
int_to_label = {i:lab for lab,i in label_to_int.items()}

# Validate labels
bad_train = sorted(set(y_train.unique()) - set(label_order))
bad_test  = sorted(set(y_test.unique()) - set(label_order))
if bad_train or bad_test:
    raise ValueError(f"Unexpected labels. train={bad_train}, test={bad_test}")

y_train_i = y_train.map(label_to_int).astype(int)
y_test_i  = y_test.map(label_to_int).astype(int)

if HAS_LGBM:
    booster = LGBMClassifier(
        objective="multiclass",
        num_class=4,
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
    )

    clf_boost = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", booster),
    ])


    def sklearn_safe_df(df: pd.DataFrame) -> pd.DataFrame:
        """Convert pandas nullable dtypes (StringDtype/Int64) that contain pd.NA into sklearn-safe object/float with np.nan."""
        out = df.copy()

        for c in out.columns:
            # If it's pandas nullable string dtype -> convert to object and replace <NA> with np.nan
            if str(out[c].dtype).startswith("string"):
                out[c] = out[c].astype(object)
                out[c] = out[c].where(pd.notna(out[c]), np.nan)

            # If it's pandas nullable Int64/Float64 -> convert to float and replace <NA> with np.nan
            if str(out[c].dtype) in ("Int64", "Float64", "boolean"):
                out[c] = out[c].astype(float)
                out[c] = out[c].where(pd.notna(out[c]), np.nan)

        # Also: if any remaining pd.NA exist anywhere, force them to np.nan
        out = out.replace({pd.NA: np.nan})
        return out

    X_train_safe = sklearn_safe_df(X_train)
    X_test_safe  = sklearn_safe_df(X_test)

    clf_boost.fit(X_train_safe, y_train_i, model__sample_weight=w_train)
    pred_i = clf_boost.predict(X_test_safe)
    pred = pd.Series(pred_i).map(int_to_label).values

    print("=== Baseline Boosted Trees (multiclass) ===")
    print(classification_report(y_test, pred, digits=4))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred))
    print("Confusion matrix labels:", label_order)
    print(confusion_matrix(y_test, pred, labels=label_order))
else:
    print("Skip boosted model (LightGBM not installed).")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001491 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1510
[LightGBM] [Info] Number of data points in the train set: 11696, number of used features: 63
[LightGBM] [Info] Start training from score -1.200667
[LightGBM] [Info] Start training from score -0.609309
[LightGBM] [Info] Start training from score -2.435258
[LightGBM] [Info] Start training from score -2.692593
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


=== Baseline Boosted Trees (multiclass) ===
               precision    recall  f1-score   support

           AD     0.6861    0.7655    0.7236       614
NoDementia_Hy     0.9231    0.9157    0.9194      2217
       PD_LBD     0.7105    0.4576    0.5567        59
     Vascular     0.0000    0.0000    0.0000        34

     accuracy                         0.8642      2924
    macro avg     0.5800    0.5347    0.5499      2924
 weighted avg     0.8584    0.8642    0.8603      2924

Balanced accuracy: 0.5346878032586538
Confusion matrix labels: ['AD', 'NoDementia_Hy', 'PD_LBD', 'Vascular']
[[ 470  139    4    1]
 [ 180 2030    6    1]
 [  18   14   27    0]
 [  17   16    1    0]]


In [7]:
# -----------------------------
# Hierarchical model
# Stage 1: NoDementia vs Dementia-within-5y
# Stage 2: subtype among {AD, PD_LBD, Vascular} conditioned on dementia
# Combine: P(subtype) = P(dementia)*P(subtype|dementia); P(NoDem) = 1 - P(dementia)
# -----------------------------
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import numpy as np
import pandas as pd
import os

if not HAS_LGBM:
    print("Hierarchical model requires LightGBM.")
else:
    # IMPORTANT: use sklearn-safe X (no pd.NA)
    X_train_safe = sklearn_safe_df(X_train)
    X_test_safe  = sklearn_safe_df(X_test)

    # -----------------------------
    # Stage 1: Binary Dementia within 5y?
    # -----------------------------
    y1_train = (y_train != "NoDementia_Hy").astype(int)
    y1_test  = (y_test  != "NoDementia_Hy").astype(int)

    # Give dementia positives higher weight
    w1_train = np.where(y1_train.values == 1, 3.0, 1.0).astype(np.float64)

    stage1 = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=800,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE
        ))
    ])

    stage1.fit(X_train_safe, y1_train, model__sample_weight=w1_train)
    p_dem = stage1.predict_proba(X_test_safe)[:, 1]  # P(dementia within 5y)

    pred1 = (p_dem >= 0.5).astype(int)

    print("\n=== Stage 1 (Binary): Dementia within 5y? ===")
    print(classification_report(y1_test, pred1, digits=4, target_names=["NoDementia", "Dementia"]))
    print("Balanced accuracy:", balanced_accuracy_score(y1_test, pred1))
    print("Confusion matrix [rows=true, cols=pred] (0=NoDementia,1=Dementia):")
    print(confusion_matrix(y1_test, pred1))

    # -----------------------------
    # -----------------------------
    # Stage 2a: Vascular vs Non-vascular among dementia cases only
    # -----------------------------
    dementia_mask_train = (y_train != "NoDementia_Hy")
    X2_train = X_train.loc[dementia_mask_train].copy()
    y2_train = y_train.loc[dementia_mask_train].copy()
    X2_train_safe = sklearn_safe_df(X2_train)

    y_vas_train = (y2_train == "Vascular").astype(int)

    # Heavier weight on vascular positives
    # w_vas = np.where(y_vas_train.values == 1, 25.0, 1.0).astype(np.float64)

    # --- Stage2a weights: balanced + optional boost ---
    n_pos = (y_vas_train == 1).sum()
    n_neg = (y_vas_train == 0).sum()

    # balanced weighting (sklearn convention): total/(2*count_class)
    w_pos = (n_pos + n_neg) / (2.0 * max(n_pos, 1))
    w_neg = (n_pos + n_neg) / (2.0 * max(n_neg, 1))

    VAS_BOOST = 1.5   # try 1.0, 1.5, 2.0, 3.0
    w_pos *= VAS_BOOST

    w_vas = np.where(y_vas_train.values == 1, w_pos, w_neg).astype(np.float64)

    print(f"[Stage2a] n_pos(Vascular)={n_pos}, n_neg(NonVascular)={n_neg}, "
        f"w_pos={w_pos:.3f}, w_neg={w_neg:.3f}")

    stage2a_vas = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE
        ))
    ])
    stage2a_vas.fit(X2_train_safe, y_vas_train, model__sample_weight=w_vas)

    # -----------------------------
    # Stage 2b: PD_LBD vs AD among NON-vascular dementia cases only
    # -----------------------------
    nonvas_mask_train = dementia_mask_train & (y_train != "Vascular")
    X3_train = X_train.loc[nonvas_mask_train].copy()
    y3_train = y_train.loc[nonvas_mask_train].copy()
    X3_train_safe = sklearn_safe_df(X3_train)

    # binary label: PD_LBD=1, AD=0
    bad = sorted(set(y3_train.unique()) - set(["AD", "PD_LBD"]))
    if bad:
        raise ValueError(f"Unexpected labels in non-vascular dementia subset: {bad}")

    y_pd_train = (y3_train == "PD_LBD").astype(int)

    # Weight PD positives a bit
    w_pd = np.where(y_pd_train.values == 1, 10.0, 1.0).astype(np.float64)

    stage2b_pd = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE
        ))
    ])
    stage2b_pd.fit(X3_train_safe, y_pd_train, model__sample_weight=w_pd)

    # -----------------------------
    # Stage 2 evaluation on TEST dementia cases (optional but useful)
    # -----------------------------
    dementia_mask_test = (y_test != "NoDementia_Hy")
    X2_test = X_test.loc[dementia_mask_test].copy()
    y2_test = y_test.loc[dementia_mask_test].copy()
    X2_test_safe = sklearn_safe_df(X2_test)

    p_vas_cond = stage2a_vas.predict_proba(X2_test_safe)[:, 1]  # P(Vascular | Dementia)

    print("\n[Stage2a TEST] p_vas_cond summary (true vascular vs others):")
    is_true_vas = (y2_test == "Vascular").values
    print("True Vascular:", pd.Series(p_vas_cond[is_true_vas]).describe())
    print("True Non-Vas:", pd.Series(p_vas_cond[~is_true_vas]).describe())

    # For non-vascular true dementia cases, compute PD vs AD
    nonvas_test_mask = dementia_mask_test & (y_test != "Vascular")
    X3_test = X_test.loc[nonvas_test_mask].copy()
    y3_test = y_test.loc[nonvas_test_mask].copy()
    X3_test_safe = sklearn_safe_df(X3_test)

    p_pd_cond_nonvas = stage2b_pd.predict_proba(X3_test_safe)[:, 1]  # P(PD | Dementia, NonVas)

    # -----------------------------
    # Combine Stage1 + Stage2a + Stage2b into 4-class prediction
    # -----------------------------
    label_order_4 = ["NoDementia_Hy", "AD", "PD_LBD", "Vascular"]
    probs = np.zeros((len(X_test_safe), 4), dtype=float)

    # Stage 2a and 2b probabilities for ALL test rows
    p_vas_all = stage2a_vas.predict_proba(X_test_safe)[:, 1]  # P(Vascular | Dementia) approx
    p_pd_all  = stage2b_pd.predict_proba(X_test_safe)[:, 1]   # P(PD | Dementia, NonVas) approx

    # Conditional subtype probabilities given dementia:
    # P(Vascular|Dem) = p_vas_all
    # P(PD|Dem) = (1 - p_vas_all) * p_pd_all
    # P(AD|Dem) = (1 - p_vas_all) * (1 - p_pd_all)
    p_ad_cond_all = (1.0 - p_vas_all) * (1.0 - p_pd_all)
    p_pd_cond_all = (1.0 - p_vas_all) * p_pd_all

    # Joint probabilities
    probs[:, 0] = 1.0 - p_dem
    probs[:, 3] = p_dem * p_vas_all
    probs[:, 2] = p_dem * p_pd_cond_all
    probs[:, 1] = p_dem * p_ad_cond_all

    # ---------- THRESHOLDS ----------
    T_DEM = 0.45
    T_VAS = 0.03
    T_PD  = 0.35
    #MARGIN_JOINT = 0.002


    pred_h = np.full(len(X_test_safe), "NoDementia_Hy", dtype=object)

    is_dem = (p_dem >= T_DEM)
    idx = np.where(is_dem)[0]
    if len(idx) > 0:
        vas_ok = (p_vas_all[idx] >= T_VAS)
        MARGIN_JOINT = -0.20
        #vas_ok = (p_vas_all[idx] >= T_VAS) & ((probs[idx, 3] - probs[idx, 1]) >= MARGIN_JOINT)

        # default AD for dementia
        pred_sub = np.full(len(idx), "AD", dtype=object)

        # among non-vascular, decide PD vs AD
        pd_ok = (p_pd_all[idx] >= T_PD)
        pred_sub[pd_ok] = "PD_LBD"
        pred_sub[vas_ok] = "Vascular"  # vascular wins if it passes its gate

        pred_h[idx] = pred_sub

    print("\n=== Hierarchical model (Stage1 + Stage2a(Vas) + Stage2b(PD)) 4-class ===")
    print(classification_report(y_test, pred_h, digits=4, labels=label_order_4))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_h))
    print("Confusion matrix labels:", label_order_4)
    print(confusion_matrix(y_test, pred_h, labels=label_order_4))

    # Save probabilities + predictions
    prob_df = pd.DataFrame(probs, columns=[f"p_{c}" for c in label_order_4], index=X_test.index)
    prob_df["p_dem"] = p_dem
    prob_df["p_ad_cond"] = p_ad_cond_all
    prob_df["p_pd_cond"] = p_pd_cond_all
    prob_df["p_vas_cond"] = p_vas_all
    prob_df["y_true"] = y_test.values
    prob_df["y_pred"] = pred_h
    prob_path = os.path.join(OUT_DIR, "hierarchical_pred_probs.csv")
    prob_df.to_csv(prob_path, index=True)
    print("Saved:", prob_path)


[LightGBM] [Info] Number of positive: 2830, number of negative: 8866
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001432 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1510
[LightGBM] [Info] Number of data points in the train set: 11696, number of used features: 63
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.489168 -> initscore=-0.043335
[LightGBM] [Info] Start training from score -0.043335


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== Stage 1 (Binary): Dementia within 5y? ===
              precision    recall  f1-score   support

  NoDementia     0.9308    0.8913    0.9106      2217
    Dementia     0.6991    0.7921    0.7427       707

    accuracy                         0.8673      2924
   macro avg     0.8149    0.8417    0.8267      2924
weighted avg     0.8748    0.8673    0.8700      2924

Balanced accuracy: 0.8416868750474507
Confusion matrix [rows=true, cols=pred] (0=NoDementia,1=Dementia):
[[1976  241]
 [ 147  560]]
[Stage2a] n_pos(Vascular)=138, n_neg(NonVascular)=2692, w_pos=15.380, w_neg=0.526
[LightGBM] [Info] Number of positive: 138, number of negative: 2692
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000480 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 854
[LightGBM] [Info] Number of data points in the train set: 2830, number of used featur

c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[Stage2a TEST] p_vas_cond summary (true vascular vs others):
True Vascular: count    34.000000
mean      0.081802
std       0.180337
min       0.000016
25%       0.001124
50%       0.008526
75%       0.065523
max       0.875403
dtype: float64
True Non-Vas: count    673.000000
mean       0.021410
std        0.083347
min        0.000004
25%        0.000113
50%        0.000547
75%        0.004448
max        0.855271
dtype: float64

=== Hierarchical model (Stage1 + Stage2a(Vas) + Stage2b(PD)) 4-class ===
               precision    recall  f1-score   support

NoDementia_Hy     0.9367    0.8809    0.9079      2217
           AD     0.6578    0.7264    0.6904       614
       PD_LBD     0.5370    0.4915    0.5133        59
     Vascular     0.1028    0.3235    0.1560        34

     accuracy                         0.8341      2924
    macro avg     0.5586    0.6056    0.5669      2924
 weighted avg     0.8604    0.8341    0.8456      2924

Balanced accuracy: 0.6055898406739908
Confusion ma

Hierarchial stage 2 updated approach without smote

In [13]:
# -----------------------------
# Hierarchical model
# Stage 1: NoDementia vs Dementia-within-5y
# Stage 2: subtype among {AD, PD_LBD, Vascular} conditioned on dementia
# Combine: P(subtype) = P(dementia)*P(subtype|dementia); P(NoDem) = 1 - P(dementia)
# -----------------------------
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import numpy as np
import pandas as pd
import os

if not HAS_LGBM:
    print("Hierarchical model requires LightGBM.")
else:
    # IMPORTANT: use sklearn-safe X (no pd.NA)
    X_train_safe = sklearn_safe_df(X_train)
    X_test_safe  = sklearn_safe_df(X_test)

    # -----------------------------
    # Stage 1: Binary Dementia within 5y?
    # -----------------------------
    y1_train = (y_train != "NoDementia_Hy").astype(int)
    y1_test  = (y_test  != "NoDementia_Hy").astype(int)

    # Give dementia positives higher weight
    w1_train = np.where(y1_train.values == 1, 3.0, 1.0).astype(np.float64)

    stage1 = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=800,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE
        ))
    ])

    stage1.fit(X_train_safe, y1_train, model__sample_weight=w1_train)
    p_dem = stage1.predict_proba(X_test_safe)[:, 1]  # P(dementia within 5y)

    pred1 = (p_dem >= 0.5).astype(int)

    print("\n=== Stage 1 (Binary): Dementia within 5y? ===")
    print(classification_report(y1_test, pred1, digits=4, target_names=["NoDementia", "Dementia"]))
    print("Balanced accuracy:", balanced_accuracy_score(y1_test, pred1))
    print("Confusion matrix [rows=true, cols=pred] (0=NoDementia,1=Dementia):")
    print(confusion_matrix(y1_test, pred1))

    # -----------------------------
    # Stage 2a: Vascular vs Non-vascular among dementia cases only
    # -----------------------------
    from sklearn.base import clone

    dementia_mask_train = (y_train != "NoDementia_Hy")
    X2_train = X_train.loc[dementia_mask_train].copy()
    y2_train = y_train.loc[dementia_mask_train].copy()
    X2_train_safe = sklearn_safe_df(X2_train)

    y_vas_train = (y2_train == "Vascular").astype(int)

    n_pos = int((y_vas_train == 1).sum())
    n_neg = int((y_vas_train == 0).sum())

    # --- LightGBM imbalance control (standard) ---
    # baseline: neg/pos ; optionally tune with VAS_BOOST
    base_spw = (n_neg / max(n_pos, 1))
    VAS_BOOST = 0.75   # try 0.75, 1.0, 1.5, 2.0
    spw = base_spw * VAS_BOOST

    print(f"[Stage2a] n_pos={n_pos}, n_neg={n_neg}, scale_pos_weight={spw:.3f}")

    stage2a_vas = Pipeline(steps=[
        ("preprocess", clone(preprocess)),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=800,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=5,
            min_child_samples=10,   # alias for min_data_in_leaf-ish behavior
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            scale_pos_weight=spw,
            random_state=RANDOM_STATE
        ))
    ])

    # NOTE: no sample_weight here (we're using scale_pos_weight instead)
    stage2a_vas.fit(X2_train_safe, y_vas_train)

    # -----------------------------
    # Stage 2b: PD_LBD vs AD among NON-vascular dementia cases only
    # -----------------------------
    nonvas_mask_train = dementia_mask_train & (y_train != "Vascular")
    X3_train = X_train.loc[nonvas_mask_train].copy()
    y3_train = y_train.loc[nonvas_mask_train].copy()
    X3_train_safe = sklearn_safe_df(X3_train)

    # binary label: PD_LBD=1, AD=0
    bad = sorted(set(y3_train.unique()) - set(["AD", "PD_LBD"]))
    if bad:
        raise ValueError(f"Unexpected labels in non-vascular dementia subset: {bad}")

    y_pd_train = (y3_train == "PD_LBD").astype(int)

    # Weight PD positives a bit
    w_pd = np.where(y_pd_train.values == 1, 10.0, 1.0).astype(np.float64)

    stage2b_pd = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE
        ))
    ])
    stage2b_pd.fit(X3_train_safe, y_pd_train, model__sample_weight=w_pd)

    # -----------------------------
    # Stage 2 evaluation on TEST dementia cases (optional but useful)
    # -----------------------------
    dementia_mask_test = (y_test != "NoDementia_Hy")
    X2_test = X_test.loc[dementia_mask_test].copy()
    y2_test = y_test.loc[dementia_mask_test].copy()
    X2_test_safe = sklearn_safe_df(X2_test)

    p_vas_cond = stage2a_vas.predict_proba(X2_test_safe)[:, 1]  # P(Vascular | Dementia)

    print("\n[Stage2a TEST] p_vas_cond summary (true vascular vs others):")
    is_true_vas = (y2_test == "Vascular").values
    print("True Vascular:", pd.Series(p_vas_cond[is_true_vas]).describe())
    print("True Non-Vas:", pd.Series(p_vas_cond[~is_true_vas]).describe())

    # For non-vascular true dementia cases, compute PD vs AD
    nonvas_test_mask = dementia_mask_test & (y_test != "Vascular")
    X3_test = X_test.loc[nonvas_test_mask].copy()
    y3_test = y_test.loc[nonvas_test_mask].copy()
    X3_test_safe = sklearn_safe_df(X3_test)

    p_pd_cond_nonvas = stage2b_pd.predict_proba(X3_test_safe)[:, 1]  # P(PD | Dementia, NonVas)

    # -----------------------------
    # Combine Stage1 + Stage2a + Stage2b into 4-class prediction
    # -----------------------------
    label_order_4 = ["NoDementia_Hy", "AD", "PD_LBD", "Vascular"]
    probs = np.zeros((len(X_test_safe), 4), dtype=float)

    # Stage 2a and 2b probabilities for ALL test rows
    p_vas_all = stage2a_vas.predict_proba(X_test_safe)[:, 1]  # P(Vascular | Dementia) approx
    p_pd_all  = stage2b_pd.predict_proba(X_test_safe)[:, 1]   # P(PD | Dementia, NonVas) approx

    # Conditional subtype probabilities given dementia:
    # P(Vascular|Dem) = p_vas_all
    # P(PD|Dem) = (1 - p_vas_all) * p_pd_all
    # P(AD|Dem) = (1 - p_vas_all) * (1 - p_pd_all)
    p_ad_cond_all = (1.0 - p_vas_all) * (1.0 - p_pd_all)
    p_pd_cond_all = (1.0 - p_vas_all) * p_pd_all

    # Joint probabilities
    probs[:, 0] = 1.0 - p_dem
    probs[:, 3] = p_dem * p_vas_all
    probs[:, 2] = p_dem * p_pd_cond_all
    probs[:, 1] = p_dem * p_ad_cond_all

    # ---------- THRESHOLDS ----------
    T_DEM = 0.45
    T_VAS = 0.08
    T_PD  = 0.35
    #MARGIN_JOINT = 0.002


    pred_h = np.full(len(X_test_safe), "NoDementia_Hy", dtype=object)

    is_dem = (p_dem >= T_DEM)
    idx = np.where(is_dem)[0]
    if len(idx) > 0:
        vas_ok = (p_vas_all[idx] >= T_VAS)
        MARGIN_JOINT = -0.20
        #vas_ok = (p_vas_all[idx] >= T_VAS) & ((probs[idx, 3] - probs[idx, 1]) >= MARGIN_JOINT)

        # default AD for dementia
        pred_sub = np.full(len(idx), "AD", dtype=object)

        # among non-vascular, decide PD vs AD
        pd_ok = (p_pd_all[idx] >= T_PD)
        pred_sub[pd_ok] = "PD_LBD"
        pred_sub[vas_ok] = "Vascular"  # vascular wins if it passes its gate

        pred_h[idx] = pred_sub

    print("\n=== Hierarchical model (Stage1 + Stage2a(Vas) + Stage2b(PD)) 4-class ===")
    print(classification_report(y_test, pred_h, digits=4, labels=label_order_4))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_h))
    print("Confusion matrix labels:", label_order_4)
    print(confusion_matrix(y_test, pred_h, labels=label_order_4))

    # Save probabilities + predictions
    prob_df = pd.DataFrame(probs, columns=[f"p_{c}" for c in label_order_4], index=X_test.index)
    prob_df["p_dem"] = p_dem
    prob_df["p_ad_cond"] = p_ad_cond_all
    prob_df["p_pd_cond"] = p_pd_cond_all
    prob_df["p_vas_cond"] = p_vas_all
    prob_df["y_true"] = y_test.values
    prob_df["y_pred"] = pred_h
    prob_path = os.path.join(OUT_DIR, "hierarchical_pred_probs.csv")
    prob_df.to_csv(prob_path, index=True)
    print("Saved:", prob_path)


[LightGBM] [Info] Number of positive: 2830, number of negative: 8866
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000919 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1510
[LightGBM] [Info] Number of data points in the train set: 11696, number of used features: 63
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.489168 -> initscore=-0.043335
[LightGBM] [Info] Start training from score -0.043335


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== Stage 1 (Binary): Dementia within 5y? ===
              precision    recall  f1-score   support

  NoDementia     0.9308    0.8913    0.9106      2217
    Dementia     0.6991    0.7921    0.7427       707

    accuracy                         0.8673      2924
   macro avg     0.8149    0.8417    0.8267      2924
weighted avg     0.8748    0.8673    0.8700      2924

Balanced accuracy: 0.8416868750474507
Confusion matrix [rows=true, cols=pred] (0=NoDementia,1=Dementia):
[[1976  241]
 [ 147  560]]
[Stage2a] n_pos=138, n_neg=2692, scale_pos_weight=14.630
[LightGBM] [Info] Number of positive: 138, number of negative: 2692
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001291 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 861
[LightGBM] [Info] Number of data points in the train set: 2830, number of used features: 62
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.048763 -> initscore=-2.970786
[L

c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


True Vascular: count    34.000000
mean      0.125985
std       0.210080
min       0.000153
25%       0.003634
50%       0.017669
75%       0.144837
max       0.855412
dtype: float64
True Non-Vas: count    673.000000
mean       0.046441
std        0.124744
min        0.000005
25%        0.000882
50%        0.004632
75%        0.028035
max        0.901184
dtype: float64

=== Hierarchical model (Stage1 + Stage2a(Vas) + Stage2b(PD)) 4-class ===
               precision    recall  f1-score   support

NoDementia_Hy     0.9367    0.8809    0.9079      2217
           AD     0.6577    0.7134    0.6844       614
       PD_LBD     0.5370    0.4915    0.5133        59
     Vascular     0.1008    0.3529    0.1569        34

     accuracy                         0.8317      2924
    macro avg     0.5581    0.6097    0.5656      2924
 weighted avg     0.8603    0.8317    0.8443      2924

Balanced accuracy: 0.6096854528602333
Confusion matrix labels: ['NoDementia_Hy', 'AD', 'PD_LBD', 'Vascular']
[[1

SMOTE Oversampling (Stage 2 Vascular)
SMOTE was applied in vascular vs non-vascular modeling.
Findings:
•	Slight vascular recall increase
•	Severe precision collapse
•	No balanced accuracy improvement
•	Increased variance
Conclusion:
Synthetic oversampling did not generate new discriminative structure and amplified noise within overlapping feature space.


In [9]:
# -----------------------------
# Hierarchical model
# Stage 1: NoDementia vs Dementia-within-5y
# Stage 2: subtype among {AD, PD_LBD, Vascular} conditioned on dementia
# Combine: P(subtype) = P(dementia)*P(subtype|dementia); P(NoDem) = 1 - P(dementia)
# -----------------------------
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import numpy as np
import pandas as pd
import os
from imblearn.over_sampling import SMOTE
from sklearn.base import clone

if not HAS_LGBM:
    print("Hierarchical model requires LightGBM.")
else:
    # IMPORTANT: use sklearn-safe X (no pd.NA)
    X_train_safe = sklearn_safe_df(X_train)
    X_test_safe  = sklearn_safe_df(X_test)

    # -----------------------------
    # Stage 1: Binary Dementia within 5y?
    # -----------------------------
    y1_train = (y_train != "NoDementia_Hy").astype(int)
    y1_test  = (y_test  != "NoDementia_Hy").astype(int)

    # Give dementia positives higher weight
    w1_train = np.where(y1_train.values == 1, 3.0, 1.0).astype(np.float64)

    stage1 = Pipeline(steps=[
        ("preprocess", clone(preprocess)),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=800,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE
        ))
    ])

    stage1.fit(X_train_safe, y1_train, model__sample_weight=w1_train)
    p_dem = stage1.predict_proba(X_test_safe)[:, 1]  # P(dementia within 5y)

    pred1 = (p_dem >= 0.5).astype(int)

    print("\n=== Stage 1 (Binary): Dementia within 5y? ===")
    print(classification_report(y1_test, pred1, digits=4, target_names=["NoDementia", "Dementia"]))
    print("Balanced accuracy:", balanced_accuracy_score(y1_test, pred1))
    print("Confusion matrix [rows=true, cols=pred] (0=NoDementia,1=Dementia):")
    print(confusion_matrix(y1_test, pred1))

    # -----------------------------
    # -----------------------------
    # Stage 2a: Vascular vs Non-vascular among dementia cases only
    # -----------------------------
    dementia_mask_train = (y_train != "NoDementia_Hy")
    X2_train = X_train.loc[dementia_mask_train].copy()
    y2_train = y_train.loc[dementia_mask_train].copy()
    X2_train_safe = sklearn_safe_df(X2_train)

    y_vas_train = (y2_train == "Vascular").astype(int)

    # 1) Fit preprocess on Stage2a TRAIN only (no leakage)
    
    pre2a = clone(preprocess)
    X2_train_mat = pre2a.fit_transform(X2_train_safe)

    # 2) SMOTE on the transformed matrix
    # target: bring vascular up to, say, 30% of non-vas
    n_pos = int((y_vas_train == 1).sum())
    n_neg = int((y_vas_train == 0).sum())

    target_pos = int(0.20 * n_neg) # try 0.20, 0.30, 0.40
    sampling_strategy = {1: target_pos}  # binary: class 1 is vascular
    sm = SMOTE(sampling_strategy=sampling_strategy, random_state=RANDOM_STATE, k_neighbors=5)

    X2_res, y2_res = sm.fit_resample(X2_train_mat, y_vas_train.values)

    print(f"[Stage2a SMOTE] before: pos={n_pos}, neg={n_neg}; after: pos={(y2_res==1).sum()}, neg={(y2_res==0).sum()}")

    # 3) Train Stage2a model on resampled matrix (NO preprocess step in pipeline now)
    stage2a_model = LGBMClassifier(
        objective="binary",
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE
    )

    stage2a_model.fit(X2_res, y2_res)

    # -----------------------------
    # Stage 2b: PD_LBD vs AD among NON-vascular dementia cases only
    # -----------------------------
    nonvas_mask_train = dementia_mask_train & (y_train != "Vascular")
    X3_train = X_train.loc[nonvas_mask_train].copy()
    y3_train = y_train.loc[nonvas_mask_train].copy()
    X3_train_safe = sklearn_safe_df(X3_train)

    # binary label: PD_LBD=1, AD=0
    bad = sorted(set(y3_train.unique()) - set(["AD", "PD_LBD"]))
    if bad:
        raise ValueError(f"Unexpected labels in non-vascular dementia subset: {bad}")

    y_pd_train = (y3_train == "PD_LBD").astype(int)

    # Weight PD positives a bit
    w_pd = np.where(y_pd_train.values == 1, 10.0, 1.0).astype(np.float64)

    stage2b_pd = Pipeline(steps=[
        ("preprocess", clone(preprocess)),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE
        ))
    ])
    stage2b_pd.fit(X3_train_safe, y_pd_train, model__sample_weight=w_pd)

    # -----------------------------
    # Stage 2 evaluation on TEST dementia cases (optional but useful)
    # -----------------------------
    dementia_mask_test = (y_test != "NoDementia_Hy")
    X2_test = X_test.loc[dementia_mask_test].copy()
    y2_test = y_test.loc[dementia_mask_test].copy()
    X2_test_safe = sklearn_safe_df(X2_test)

    X2_test_mat = pre2a.transform(X2_test_safe)
    p_vas_cond = stage2a_model.predict_proba(X2_test_mat)[:, 1]

    print("\n[Stage2a TEST] p_vas_cond summary (true vascular vs others):")
    is_true_vas = (y2_test == "Vascular").values
    print("True Vascular:", pd.Series(p_vas_cond[is_true_vas]).describe())
    print("True Non-Vas:", pd.Series(p_vas_cond[~is_true_vas]).describe())

    # For non-vascular true dementia cases, compute PD vs AD
    nonvas_test_mask = dementia_mask_test & (y_test != "Vascular")
    X3_test = X_test.loc[nonvas_test_mask].copy()
    y3_test = y_test.loc[nonvas_test_mask].copy()
    X3_test_safe = sklearn_safe_df(X3_test)

    p_pd_cond_nonvas = stage2b_pd.predict_proba(X3_test_safe)[:, 1]  # P(PD | Dementia, NonVas)

    # -----------------------------
    # Combine Stage1 + Stage2a + Stage2b into 4-class prediction
    # -----------------------------
    label_order_4 = ["NoDementia_Hy", "AD", "PD_LBD", "Vascular"]
    probs = np.zeros((len(X_test_safe), 4), dtype=float)

    X2_test_mat = pre2a.transform(X2_test_safe)

    X_test_mat = pre2a.transform(X_test_safe)
    p_vas_all = stage2a_model.predict_proba(X_test_mat)[:, 1]

    # Conditional subtype probabilities given dementia:
    # P(Vascular|Dem) = p_vas_all
    # P(PD|Dem) = (1 - p_vas_all) * p_pd_all
    # P(AD|Dem) = (1 - p_vas_all) * (1 - p_pd_all)

    p_pd_all = stage2b_pd.predict_proba(X_test_safe)[:, 1]
    p_ad_cond_all = (1.0 - p_vas_all) * (1.0 - p_pd_all)
    p_pd_cond_all = (1.0 - p_vas_all) * p_pd_all

    # Joint probabilities
    probs[:, 0] = 1.0 - p_dem
    probs[:, 3] = p_dem * p_vas_all
    probs[:, 2] = p_dem * p_pd_cond_all
    probs[:, 1] = p_dem * p_ad_cond_all

    # ---------- THRESHOLDS ----------
    
    T_DEM = 0.35
    T_VAS = 0.03
    T_PD  = 0.35
    #MARGIN_JOINT = 0.002


    pred_h = np.full(len(X_test_safe), "NoDementia_Hy", dtype=object)

    is_dem = (p_dem >= T_DEM)
    idx = np.where(is_dem)[0]
    if len(idx) > 0:
        vas_ok = (p_vas_all[idx] >= T_VAS)
        MARGIN_JOINT = -0.20
        #vas_ok = (p_vas_all[idx] >= T_VAS) & ((probs[idx, 3] - probs[idx, 1]) >= MARGIN_JOINT)

        # default AD for dementia
        pred_sub = np.full(len(idx), "AD", dtype=object)

        # among non-vascular, decide PD vs AD
        pd_ok = (p_pd_all[idx] >= T_PD)
        pred_sub[pd_ok] = "PD_LBD"
        pred_sub[vas_ok] = "Vascular"  # vascular wins if it passes its gate

        pred_h[idx] = pred_sub

    print("\n=== Hierarchical model (Stage1 + Stage2a(Vas) + Stage2b(PD)) 4-class ===")
    print(classification_report(y_test, pred_h, digits=4, labels=label_order_4))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_h))
    print("Confusion matrix labels:", label_order_4)
    print(confusion_matrix(y_test, pred_h, labels=label_order_4))

    # Save probabilities + predictions
    prob_df = pd.DataFrame(probs, columns=[f"p_{c}" for c in label_order_4], index=X_test.index)
    prob_df["p_dem"] = p_dem
    prob_df["p_ad_cond"] = p_ad_cond_all
    prob_df["p_pd_cond"] = p_pd_cond_all
    prob_df["p_vas_cond"] = p_vas_all
    prob_df["y_true"] = y_test.values
    prob_df["y_pred"] = pred_h
    prob_path = os.path.join(OUT_DIR, "hierarchical_pred_probs.csv")
    prob_df.to_csv(prob_path, index=True)
    print("Saved:", prob_path)


[LightGBM] [Info] Number of positive: 2830, number of negative: 8866
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001320 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1510
[LightGBM] [Info] Number of data points in the train set: 11696, number of used features: 63
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.489168 -> initscore=-0.043335
[LightGBM] [Info] Start training from score -0.043335


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== Stage 1 (Binary): Dementia within 5y? ===
              precision    recall  f1-score   support

  NoDementia     0.9308    0.8913    0.9106      2217
    Dementia     0.6991    0.7921    0.7427       707

    accuracy                         0.8673      2924
   macro avg     0.8149    0.8417    0.8267      2924
weighted avg     0.8748    0.8673    0.8700      2924

Balanced accuracy: 0.8416868750474507
Confusion matrix [rows=true, cols=pred] (0=NoDementia,1=Dementia):
[[1976  241]
 [ 147  560]]
[Stage2a SMOTE] before: pos=138, neg=2692; after: pos=538, neg=2692
[LightGBM] [Info] Number of positive: 538, number of negative: 2692
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004500 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3208
[LightGBM] [Info] Number of data points in the train set: 3230, number of used features: 61
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.166563 -> initscore=

c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[Stage2a TEST] p_vas_cond summary (true vascular vs others):
True Vascular: count    34.000000
mean      0.040274
std       0.092012
min       0.000013
25%       0.000874
50%       0.003760
75%       0.020302
max       0.383808
dtype: float64
True Non-Vas: count    673.000000
mean       0.013535
std        0.068436
min        0.000003
25%        0.000078
50%        0.000387
75%        0.002844
max        0.889883
dtype: float64


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== Hierarchical model (Stage1 + Stage2a(Vas) + Stage2b(PD)) 4-class ===
               precision    recall  f1-score   support

NoDementia_Hy     0.9465    0.8624    0.9025      2217
           AD     0.6221    0.7883    0.6954       614
       PD_LBD     0.5179    0.4915    0.5043        59
     Vascular     0.1000    0.2059    0.1346        34

     accuracy                         0.8317      2924
    macro avg     0.5466    0.5870    0.5592      2924
 weighted avg     0.8599    0.8317    0.8421      2924

Balanced accuracy: 0.5870270237641588
Confusion matrix labels: ['NoDementia_Hy', 'AD', 'PD_LBD', 'Vascular']
[[1912  262   13   30]
 [  86  484   13   31]
 [  11   17   29    2]
 [  11   15    1    7]]
Saved: ../output\hierarchical_pred_probs.csv


In [71]:
print("T_VAS:", T_VAS, "MARGIN_JOINT:", MARGIN_JOINT)
print("Eligible by p_vas_all >= T_VAS:", int(np.sum(p_vas_all[idx] >= T_VAS)))

d = (probs[idx, 3] - probs[idx, 1])
eligible = (p_vas_all[idx] >= T_VAS)
print("Eligible by margin among eligible:", int(np.sum(eligible & (d >= MARGIN_JOINT))))
print("Joint diff stats among eligible: min/median/max =",
      float(np.min(d[eligible])) if np.any(eligible) else None,
      float(np.median(d[eligible])) if np.any(eligible) else None,
      float(np.max(d[eligible])) if np.any(eligible) else None)


T_VAS: 0.03 MARGIN_JOINT: -0.2
Eligible by p_vas_all >= T_VAS: 92
Eligible by margin among eligible: 22
Joint diff stats among eligible: min/median/max = -0.9288056286745079 -0.5614181954530882 0.6344708615528053


In [72]:

# -----------------------------
# Quick "which factors matter" (model-driven)
# For LightGBM: built-in gain importance on the multiclass baseline model
# -----------------------------
if HAS_LGBM:
    # Get feature names after preprocessing
    def get_feature_names(ct: ColumnTransformer):
        out = []
        # numeric
        out.extend(ct.named_transformers_["num"].get_feature_names_out(num_cols))
        # categorical
        ohe = ct.named_transformers_["cat"].named_steps["onehot"]
        cats = ohe.get_feature_names_out(cat_cols)
        out.extend(cats.tolist())
        return out

    feat_names = get_feature_names(clf_boost.named_steps["preprocess"])
    booster_model = clf_boost.named_steps["model"]

    # LightGBM importance is aggregated across trees/classes
    imp = pd.DataFrame({
        "feature": feat_names,
        "importance_gain": booster_model.feature_importances_
    }).sort_values("importance_gain", ascending=False)

    imp_path = os.path.join(OUT_DIR, "feature_importance_boosted.csv")
    imp.to_csv(imp_path, index=False)
    print("Saved:", imp_path)
    display(imp.head(25))


Saved: ../output\feature_importance_boosted.csv


,feature,importance_gain
27,TRAILB,41902
26,TRAILA,39022
0,NACCAGE,36688
28,ANIMALS,29293
1,EDUC,20552
29,BOSTON,18340
30,CDRSUM,17622
24,NACCMMSE,13999
25,NACCMOCA,10346
5,HYPERCHO,4940


In [73]:

# -----------------------------
# Subgroup performance (sex/race/education bin)
# Note: subgroup counts for rare classes can be tiny -> interpret cautiously.
# -----------------------------
from sklearn.metrics import balanced_accuracy_score

def subgroup_report(predict_fn, X_test_df, y_test_ser, group_series, group_name):
    rows = []
    for g in sorted(group_series.dropna().unique()):
        mask = (group_series == g)
        if mask.sum() < 30:
            continue
        yt = y_test_ser[mask]
        yp = predict_fn(X_test_df.loc[mask])
        rows.append({
            "group_name": group_name,
            "group_value": str(g),
            "n": int(mask.sum()),
            "balanced_acc": balanced_accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro"),
            "recall_AD": ( (yp == "AD") & (yt == "AD") ).sum() / max(1, (yt == "AD").sum()),
            "recall_NoDementia": ( (yp == "NoDementia_Hy") & (yt == "NoDementia_Hy") ).sum() / max(1, (yt == "NoDementia_Hy").sum()),
            "recall_PD_LBD": ( (yp == "PD_LBD") & (yt == "PD_LBD") ).sum() / max(1, (yt == "PD_LBD").sum()),
            "recall_Vascular": ( (yp == "Vascular") & (yt == "Vascular") ).sum() / max(1, (yt == "Vascular").sum()),
        })
    return pd.DataFrame(rows)

subgroups = {}
for col in ["SEX","RACE","EDUC_BIN"]:
    if col in X_test.columns:
        subgroups[col] = X_test[col]

tables = []
if HAS_LGBM:
    # predict wrapper for baseline boosted model
    def predict_boost(Xdf):
        Xdf_safe = sklearn_safe_df(Xdf)          # <-- key fix
        pi = clf_boost.predict(Xdf_safe)
        return pd.Series(pi).map(int_to_label).values


    for name, s in subgroups.items():
        tables.append(subgroup_report(predict_boost, X_test, y_test, s, name))
    sub_boost = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
    p_boost = os.path.join(OUT_DIR, "subgroup_metrics_boosted.csv")
    sub_boost.to_csv(p_boost, index=False)
    print("Saved:", p_boost)
    display(sub_boost)


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have v

Saved: ../output\subgroup_metrics_boosted.csv


,group_name,group_value,n,balanced_acc,macro_f1,recall_AD,recall_NoDementia,recall_PD_LBD,recall_Vascular
0,SEX,1,1237,0.279114,0.263151,0.099359,0.995360,0.021739,0.0
1,SEX,2,1687,0.287431,0.290982,0.155629,0.994096,0.000000,0.0
2,RACE,1,2427,0.284424,0.280496,0.125000,0.994515,0.018182,0.0
3,RACE,2,367,0.269525,0.263829,0.084746,0.993355,0.000000,0.0
4,RACE,5,79,0.385965,0.379466,0.157895,1.000000,0.000000,0.0
5,EDUC_BIN,<=12,478,0.315676,0.311817,0.275362,0.987342,0.000000,0.0
6,EDUC_BIN,>12,2446,0.275391,0.268961,0.084034,0.995792,0.021739,0.0


## SHAP explanations (tree models)
This section explains which transformed features drive each stage of the hierarchical model. It will run only if `shap` is installed and LightGBM models are available.

In [74]:
# -----------------------------
# SHAP explanations for Stage 1 / Stage 2a / Stage 2b
# Works for LightGBM (tree) models. Skips gracefully if shap isn't installed.
# -----------------------------
import numpy as np
import os

try:
    import shap
    import matplotlib.pyplot as plt
    HAS_SHAP = True
except Exception as e:
    HAS_SHAP = False
    print("SHAP not available:", repr(e))

def save_current_fig(out_dir: str, filename: str):
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, filename)
    plt.gcf().savefig(path, dpi=200, bbox_inches="tight")
    plt.close(plt.gcf())
    print("Saved SHAP figure:", path)

def _get_feature_names_from_preprocess(preprocess, X_df):
    """Best-effort feature name extraction for a fitted ColumnTransformer."""
    try:
        return preprocess.get_feature_names_out()
    except Exception:
        pass
    # Fallback: build generic names
    return np.array([f"f{i}" for i in range(preprocess.transform(X_df).shape[1])], dtype=object)

def _shap_summary_for_pipeline(pipe, X_ref_df, title, save_name, out_dir=FIG_DIR, max_display=30):
    """
    Computes SHAP values on the *model input space* (after preprocess transform)
    and shows a summary plot + saves it as PNG.
    """
    if not HAS_SHAP:
        return
    if pipe is None:
        print(f"Skip SHAP ({title}): model is None")
        return
    if "preprocess" not in pipe.named_steps or "model" not in pipe.named_steps:
        print(f"Skip SHAP ({title}): pipeline missing preprocess/model")
        return

    pre = pipe.named_steps["preprocess"]
    mdl = pipe.named_steps["model"]

    # Transform once
    X_mat = pre.transform(X_ref_df)
    feat_names = _get_feature_names_from_preprocess(pre, X_ref_df)

    # TreeExplainer works for LGBMClassifier
    try:
        explainer = shap.TreeExplainer(mdl)
        shap_values = explainer.shap_values(X_mat)
    except Exception as e:
        print(f"Skip SHAP ({title}):", repr(e))
        return

    print(f"\n[SHAP] {title}:")
    # Binary -> array; Multiclass -> list of arrays
    if isinstance(shap_values, list):
        # For multiclass (if you ever use it), plot mean(|shap|) across classes
        vals = np.mean([np.abs(v) for v in shap_values], axis=0)
        shap.summary_plot(vals, X_mat, feature_names=feat_names, max_display=max_display, show=False)
        save_current_fig(out_dir, save_name)
    else:
        shap.summary_plot(shap_values, X_mat, feature_names=feat_names, max_display=max_display, show=False)
        save_current_fig(out_dir, save_name)

# Run SHAP on the test set (or a sample if it's large)
if HAS_LGBM and HAS_SHAP:
    # Use a capped sample for speed
    SAMPLE_N = 2000
    if len(X_test_safe) > SAMPLE_N:
        samp_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test_safe), size=SAMPLE_N, replace=False)
        X_shap = X_test_safe.iloc[samp_idx].copy()
    else:
        X_shap = X_test_safe.copy()

    _shap_summary_for_pipeline(
        stage1, X_shap,
        title="Stage 1: Dementia within 5y (binary)",
        save_name="shap_hier_stage1_summary.png",
        max_display=25
    )
    _shap_summary_for_pipeline(
        stage2a_vas, X_shap,
        title="Stage 2a: Vascular vs non-vascular (binary)",
        save_name="shap_hier_stage2a_vascular_summary.png",
        max_display=25
    )
    _shap_summary_for_pipeline(
        stage2b_pd, X_shap,
        title="Stage 2b: PD/LBD vs AD (binary)",
        save_name="shap_hier_stage2b_pd_vs_ad_summary.png",
        max_display=25
    )

c:\Python314\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



[SHAP] Stage 1: Dementia within 5y (binary):
Saved SHAP figure: ../output\figs_hierarchical\shap_hier_stage1_summary.png


c:\Python314\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



[SHAP] Stage 2a: Vascular vs non-vascular (binary):
Saved SHAP figure: ../output\figs_hierarchical\shap_hier_stage2a_vascular_summary.png


c:\Python314\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



[SHAP] Stage 2b: PD/LBD vs AD (binary):
Saved SHAP figure: ../output\figs_hierarchical\shap_hier_stage2b_pd_vs_ad_summary.png


In [75]:
import os
import numpy as np

# numpy 2.x compatibility for older shap code paths
if not hasattr(np, "trapz") and hasattr(np, "trapezoid"):
    np.trapz = np.trapezoid
if not hasattr(np, "in1d") and hasattr(np, "isin"):
    np.in1d = np.isin

import shap
import matplotlib.pyplot as plt

def _ensure_dir(d):
    os.makedirs(d, exist_ok=True)

def _get_feat_names(pre, X_df):
    try:
        return pre.get_feature_names_out()
    except Exception:
        # best effort fallback
        return np.array([f"f{i}" for i in range(pre.transform(X_df).shape[1])], dtype=object)

def _pick_row_idx(pipe, X_df, target_positive=True):
    """Pick one representative row: highest prob for positive if target_positive else lowest."""
    pre = pipe.named_steps["preprocess"]
    mdl = pipe.named_steps["model"]
    X_mat = pre.transform(X_df)
    proba = mdl.predict_proba(X_mat)[:, 1]
    return int(np.argmax(proba) if target_positive else np.argmin(proba))

def save_waterfall_for_binary_pipeline(pipe, X_df, title, out_path, row_idx=None, max_display=20):
    """
    Saves a SHAP waterfall for one row for a binary tree model inside a pipeline.
    Uses TreeExplainer for LightGBM.
    """
    _ensure_dir(os.path.dirname(out_path))

    pre = pipe.named_steps["preprocess"]
    mdl = pipe.named_steps["model"]
    feat_names = _get_feat_names(pre, X_df)

    X_mat = pre.transform(X_df)
    if row_idx is None:
        row_idx = _pick_row_idx(pipe, X_df, target_positive=True)

    explainer = shap.TreeExplainer(mdl)
    sv = explainer(X_mat)  # Explanation

    # For binary models, shap returns sv.values shape (n, p) and base_values shape (n,)
    # Waterfall expects a single Explanation for one sample
    one = shap.Explanation(
        values=sv.values[row_idx],
        base_values=sv.base_values[row_idx] if np.ndim(sv.base_values) else sv.base_values,
        data=X_mat[row_idx].toarray().ravel() if hasattr(X_mat[row_idx], "toarray") else np.array(X_mat[row_idx]).ravel(),
        feature_names=list(feat_names)
    )

    plt.figure(figsize=(9, 6))
    shap.plots.waterfall(one, max_display=max_display, show=False)
    plt.title(title)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", out_path, "| row_idx:", row_idx)

# Example usage on your SHAP sample (X_shap) and your FIG_DIR
# Make sure FIG_DIR exists and X_shap is defined like you already do
_ensure_dir(FIG_DIR)

# ---- Stage 1 waterfall ----
# Positive side (likely dementia)
save_waterfall_for_binary_pipeline(
    stage1, X_shap,
    title="Stage 1 waterfall: dementia vs no dementia (example high-risk)",
    out_path=os.path.join(FIG_DIR, "shap_waterfall_stage1_pred_Dementia.png"),
    row_idx=None,   # auto-pick max proba
    max_display=25
)
# Negative side (low-risk / no dementia)
save_waterfall_for_binary_pipeline(
    stage1, X_shap,
    title="Stage 1 waterfall: dementia vs no dementia (example low-risk)",
    out_path=os.path.join(FIG_DIR, "shap_waterfall_stage1_pred_NoDementia_Hy.png"),
    row_idx=_pick_row_idx(stage1, X_shap, target_positive=False),
    max_display=25
)

# ---- Stage 2a waterfall (only makes sense on dementia-like cases) ----
# If you want, restrict X_shap to those predicted dementia by stage1 before explaining stage2a
# Here we just run on X_shap; you can filter for better semantics.
save_waterfall_for_binary_pipeline(
    stage2a_vas, X_shap,
    title="Stage 2a waterfall: vascular vs non-vascular (example vascular-leaning)",
    out_path=os.path.join(FIG_DIR, "shap_waterfall_stage2a_pred_Vascular.png"),
    row_idx=None,
    max_display=25
)

# ---- Stage 2b waterfall (PD/LBD vs AD among non-vascular dementia) ----
save_waterfall_for_binary_pipeline(
    stage2b_pd, X_shap,
    title="Stage 2b waterfall: PD/LBD vs AD (example PD/LBD-leaning)",
    out_path=os.path.join(FIG_DIR, "shap_waterfall_stage2b_pred_PD_LBD.png"),
    row_idx=None,
    max_display=25
)


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Saved: ../output\figs_hierarchical\shap_waterfall_stage1_pred_Dementia.png | row_idx: 1995


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Saved: ../output\figs_hierarchical\shap_waterfall_stage1_pred_NoDementia_Hy.png | row_idx: 1200


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Saved: ../output\figs_hierarchical\shap_waterfall_stage2a_pred_Vascular.png | row_idx: 1862


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Saved: ../output\figs_hierarchical\shap_waterfall_stage2b_pred_PD_LBD.png | row_idx: 575


In [76]:
# print("CWD:", os.getcwd())
# print("HAS_LGBM:", HAS_LGBM, "HAS_SHAP:", HAS_SHAP)
# print("FIG_DIR abs:", os.path.abspath(FIG_DIR))

# # ... later, change show=True -> show=False
# # shap.summary_plot(..., show=False)


## Feature-group ablation study
This evaluates how performance changes when the model can only use (a) demographics, (b) demographics+medical, (c) demographics+medical+cognition, vs (d) full feature set. For fairness, each setting re-trains the full hierarchical pipeline.

In [77]:

# -----------------------------
# Ablation study (feature groups) for hierarchical model
# -----------------------------
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score

def _cast_code_categoricals(df_in, cols):
    """Convert selected coded columns to object strings; keep NaN as NaN."""
    df = df_in.copy()
    for c in cols:
        if c in df.columns:
            df[c] = df[c].map(lambda v: np.nan if pd.isna(v) else str(v)).astype(object)
    return df

def train_eval_hierarchical_on_cols(cols, T_DEM=0.45, T_PD=0.35, T_VAS=0.03,
                                   w_dem_pos=3.0, w_vas_pos=25.0, w_pd_pos=10.0,
                                   random_state=RANDOM_STATE):
    """
    Train Stage1 + Stage2a + Stage2b on a restricted column set and evaluate 4-class output.
    Returns summary metrics (macro_f1, bal_acc, vascular_recall, vascular_precision).
    """
    # Subset
    Xtr = X_train[cols].copy()
    Xte = X_test[cols].copy()

    # Treat common coded categoricals as categorical strings
    coded_cat = [c for c in ["SEX", "RACE", "HISPANIC", "NACCUDSD", "EDUC", "EDUC_BIN"] if c in cols]
    Xtr = _cast_code_categoricals(Xtr, coded_cat)
    Xte = _cast_code_categoricals(Xte, coded_cat)

    # sklearn-safe
    Xtr = sklearn_safe_df(Xtr)
    Xte = sklearn_safe_df(Xte)

    # Build a preprocess just for these cols
    cat_cols_ = [c for c in Xtr.columns if Xtr[c].dtype == "object"]
    num_cols_ = [c for c in Xtr.columns if c not in cat_cols_]

    pre_ = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols_),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols_),
        ],
        remainder="drop"
    )

    if not HAS_LGBM:
        raise RuntimeError("Ablation requires LightGBM in this notebook.")

    # Stage 1
    y1_tr = (y_train != "NoDementia_Hy").astype(int)
    y1_te = (y_test  != "NoDementia_Hy").astype(int)
    w1 = np.where(y1_tr.values == 1, float(w_dem_pos), 1.0).astype(np.float64)

    s1 = Pipeline(steps=[
        ("preprocess", pre_),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=800,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=random_state
        ))
    ])
    s1.fit(Xtr, y1_tr, model__sample_weight=w1)
    p_dem_local = s1.predict_proba(Xte)[:, 1]
    is_dem = (p_dem_local >= T_DEM)

    # Stage 2a (Vascular vs rest) among dementia training rows
    dem_tr_mask = (y_train != "NoDementia_Hy")
    X2_tr = Xtr.loc[dem_tr_mask].copy()
    y2_tr = y_train.loc[dem_tr_mask].copy()
    y_vas_tr = (y2_tr == "Vascular").astype(int)
    w_v = np.where(y_vas_tr.values == 1, float(w_vas_pos), 1.0).astype(np.float64)

    s2a = Pipeline(steps=[
        ("preprocess", pre_),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=random_state
        ))
    ])
    s2a.fit(X2_tr, y_vas_tr, model__sample_weight=w_v)

    # Stage 2b (PD vs AD) among non-vascular dementia training rows
    nonvas_tr_mask = (y_train != "NoDementia_Hy") & (y_train != "Vascular")
    X3_tr = Xtr.loc[nonvas_tr_mask].copy()
    y3_tr = y_train.loc[nonvas_tr_mask].copy()
    # map: PD=1, AD=0 (ignore anything else; should already be filtered by KEEP)
    y_pd_tr = (y3_tr == "PD_LBD").astype(int)
    w_p = np.where(y_pd_tr.values == 1, float(w_pd_pos), 1.0).astype(np.float64)

    s2b = Pipeline(steps=[
        ("preprocess", pre_),
        ("model", LGBMClassifier(
            objective="binary",
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=random_state
        ))
    ])
    s2b.fit(X3_tr, y_pd_tr, model__sample_weight=w_p)

    # Combine predictions
    p_vas = s2a.predict_proba(Xte)[:, 1]
    p_pd  = s2b.predict_proba(Xte)[:, 1]

    pred = np.full(len(Xte), "NoDementia_Hy", dtype=object)
    idx = np.where(is_dem)[0]
    if len(idx) > 0:
        # subtype defaults
        sub = np.full(len(idx), "AD", dtype=object)
        vas_ok = (p_vas[idx] >= T_VAS)
        pd_ok  = (p_pd[idx]  >= T_PD)
        sub[pd_ok] = "PD_LBD"
        sub[vas_ok] = "Vascular"  # vascular wins if passes its threshold
        pred[idx] = sub

    # Metrics
    labels4 = ["NoDementia_Hy","AD","PD_LBD","Vascular"]
    macro_f1 = f1_score(y_test, pred, labels=labels4, average="macro")
    bal_acc = balanced_accuracy_score(y_test, pred)

    # vascular precision/recall
    # avoid sklearn warnings by computing manually
    y_true = np.array(y_test)
    y_pred = np.array(pred)
    vas_tp = np.sum((y_true == "Vascular") & (y_pred == "Vascular"))
    vas_fp = np.sum((y_true != "Vascular") & (y_pred == "Vascular"))
    vas_fn = np.sum((y_true == "Vascular") & (y_pred != "Vascular"))
    vas_prec = vas_tp / (vas_tp + vas_fp) if (vas_tp + vas_fp) else 0.0
    vas_rec  = vas_tp / (vas_tp + vas_fn) if (vas_tp + vas_fn) else 0.0

    return {
        "cols": len(cols),
        "macro_f1": macro_f1,
        "bal_acc": bal_acc,
        "vas_prec": vas_prec,
        "vas_rec": vas_rec,
    }

# Define feature groups (same spirit as the baseline notebook; adjust if your columns differ)
DEMOG = [c for c in ["NACCAGE","SEX","RACE","HISPANIC","EDUC","NACCUDSD"] if c in X_all.columns]
MED = [c for c in [
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
    "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",
    "APNEA","DEP2YRS",
    "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD",
] if c in X_all.columns]
COG_FUNC = [c for c in ["NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON","CDRSUM","CDRGLOB"] if c in X_all.columns]

ABLATIONS = {
    "Demographics": DEMOG,
    "Demog+Medical": sorted(set(DEMOG + MED)),
    "Demog+Medical+Cognition": sorted(set(DEMOG + MED + COG_FUNC)),
    "Full": list(X_all.columns),
}

print("\n=== Ablation study (hierarchical) ===")
rows = []
for name, cols in ABLATIONS.items():
    if len(cols) < 3:
        continue
    out = train_eval_hierarchical_on_cols(cols, T_DEM=T_DEM, T_PD=T_PD, T_VAS=T_VAS,
                                          w_dem_pos=3.0, w_vas_pos=25.0, w_pd_pos=10.0)
    out["setting"] = name
    rows.append(out)

ab_df = pd.DataFrame(rows).set_index("setting").sort_values("macro_f1", ascending=False)
display(ab_df)



=== Ablation study (hierarchical) ===
[LightGBM] [Info] Number of positive: 2830, number of negative: 8866
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000486 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 138
[LightGBM] [Info] Number of data points in the train set: 11696, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.489168 -> initscore=-0.043335
[LightGBM] [Info] Start training from score -0.043335
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 138, number of negative: 2692
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000770 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 100
[LightGBM] [Info] Number of data points in the train set: 2830, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.561706 -> initscore=0.248090
[LightGBM] [Info] Start training from score 0.248090
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2830, number of negative: 8866
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003010 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 193
[LightGBM] [Info] Number of data points in the train set: 11696, number of used features: 53
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.489168 -> initscore=-0.043335
[LightGBM] [Info] Start training from score -0.043335
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


...

[LightGBM] [Info] Number of positive: 2830, number of negative: 8866
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002231 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1462
[LightGBM] [Info] Number of data points in the train set: 11696, number of used features: 70
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.489168 -> initscore=-0.043335
[LightGBM] [Info] Start training from score -0.043335


c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 138, number of negative: 2692
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003597 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 800
[LightGBM] [Info] Number of data points in the train set: 2830, number of used features: 61
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.561706 -> initscore=0.248090
[LightGBM] [Info] Start training from score 0.248090
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,cols,macro_f1,bal_acc,vas_prec,vas_rec
setting,,,,,
Full,44,0.498587,0.515735,0.140000,0.205882
Demog+Medical+Cognition,35,0.480391,0.509457,0.104651,0.264706
Demog+Medical,27,0.444570,0.523005,0.082353,0.411765
Demographics,6,0.361610,0.441954,0.028384,0.382353


In [78]:
# -----------------------------
# Export hierarchical run summaries (no overwrite with baseline)
# -----------------------------
import os
import pandas as pd

# Save ablation table
ablation_path = os.path.join(OUT_DIR, "ablation_hierarchical.csv")
try:
    ab_df.to_csv(ablation_path)
    print("Saved:", ablation_path)
except Exception as e:
    print("Could not save ablation table:", e)

# Build a compact run summary for easy comparison with baseline notebook
summary_rows = []
if isinstance(ab_df, pd.DataFrame) and len(ab_df) > 0:
    # Prefer the 'Full' row if present; otherwise take the best macro_f1 row
    if "Full" in ab_df.index:
        r = ab_df.loc["Full"].to_dict()
        r["setting"] = "Full"
    else:
        best_idx = ab_df["macro_f1"].astype(float).idxmax() if "macro_f1" in ab_df.columns else ab_df.index[0]
        r = ab_df.loc[best_idx].to_dict()
        r["setting"] = str(best_idx)
    # Add key hyperparameters/thresholds for traceability
    r.update({
        "T_DEM": float(T_DEM),
        "T_PD": float(T_PD),
        "T_VAS": float(T_VAS),
        "w_dem_pos": 3.0,
        "w_vas_pos": 25.0,
        "w_pd_pos": 10.0,
        "artifact_ablation": ablation_path,
        "artifact_pred_probs": os.path.join(OUT_DIR, "hierarchical_pred_probs.csv"),
    })
    summary_rows.append(r)

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUT_DIR, "run_summary_hierarchical.csv")
summary_df.to_csv(summary_path, index=False)
print("Saved:", summary_path)
display(summary_df)


Saved: ../output\ablation_hierarchical.csv
Saved: ../output\run_summary_hierarchical.csv


,cols,macro_f1,bal_acc,vas_prec,vas_rec,setting,T_DEM,T_PD,T_VAS,w_dem_pos,w_vas_pos,w_pd_pos,artifact_ablation,artifact_pred_probs
0,44.0,0.498587,0.515735,0.14,0.205882,Full,0.45,0.35,0.03,3.0,25.0,10.0,../output\ablation_hierarchical.csv,../output\hierarchical_pred_probs.csv
